# Batch Analysis & Model Training

## Objective

The objective of this notebook is to prepare historical transaction data from the PaySim dataset for fraud detection model training.

In this notebook, we will:

- Load the historical transaction dataset
- Explore the dataset structure
- Check data quality
- Analyze fraud distribution
- Perform preprocessing
- Prepare features for anomaly detection models

In [1]:
import pandas as pd
import numpy as np

# Display all columns
pd.set_option("display.max_columns", None)

# Dataset path
DATA_PATH = "../data/PS_20174392719_1491204439457_log.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")

Dataset loaded successfully!


## Preview of Dataset

Let's inspect the first few records to understand the structure of the PaySim dataset.

In [2]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## Dataset Dimensions

Checking the total number of rows and columns.

In [3]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Rows : 6362620
Columns : 11


## Dataset Information

Understanding:

- Data types
- Missing values
- Memory usage

In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 706.2 MB


## Statistical Summary

Generating descriptive statistics for all numerical features.

In [5]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


## Missing Value Analysis

Checking whether the dataset contains missing values.

In [6]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## Fraud Distribution

Understanding how many fraudulent and non-fraudulent transactions exist.

In [7]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

## Fraud Percentage

Calculating the percentage of fraudulent transactions in the dataset.

In [8]:
fraud_percentage = df["isFraud"].mean() * 100

print(f"Fraud Percentage : {fraud_percentage:.4f}%")

Fraud Percentage : 0.1291%


## Transaction Types

Exploring the different transaction categories available in the dataset.

In [9]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

## Fraud by Transaction Type

Analyzing which transaction types are associated with fraud.

In [10]:
pd.crosstab(df["type"], df["isFraud"])

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


## Summary

### Key Observations

- Dataset successfully loaded.
- Fraud cases represent a very small percentage of total transactions.
- No significant missing values were observed.
- Multiple transaction categories are present.
- Fraud is concentrated in specific transaction types.

The dataset is now ready for preprocessing and feature engineering in the next stage.

# Data Preprocessing

## Objective

Before training a machine learning model, the dataset must be preprocessed to ensure data quality and compatibility with machine learning algorithms.

### Steps

- Check the dataset structure
- Remove unnecessary columns
- Encode categorical variables
- Separate features and target variable

In [11]:
# Display dataset information

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 706.2 MB


In [12]:
# Check number of rows and columns
print("Dataset Shape:", df.shape)

Dataset Shape: (6362620, 11)


In [13]:
# Display data types
print(df.dtypes)

step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


## Data Preprocessing

Before training a machine learning model, the dataset must be cleaned and transformed into a suitable format. In this section, we will remove unnecessary columns, encode categorical variables, and prepare the dataset for model training.

# Remove Unnecessary Columns

Some columns such as transaction IDs and account names are unique identifiers and do not contribute to fraud prediction. These columns are removed to improve model performance and reduce unnecessary complexity.

In [14]:
# Remove identifier columns

df = df.drop(columns=["nameOrig", "nameDest"])

print("Columns after removing identifiers:")
print(df.columns)

Columns after removing identifiers:
Index(['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
       'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud'],
      dtype='str')


In [15]:
# Check missing values

missing_values = df.isnull().sum()

print(missing_values)

step              0
type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [16]:
# Check duplicate rows

duplicates = df.duplicated().sum()

print("Duplicate Records:", duplicates)

Duplicate Records: 543


In [17]:
# Remove duplicate rows

df.drop_duplicates(inplace=True)

print("Dataset Shape after removing duplicates:")
print(df.shape)

Dataset Shape after removing duplicates:
(6362077, 9)


## Summary

- Removed identifier columns (`nameOrig`, `nameDest`)
- Verified missing values
- Checked and removed duplicate records
- Dataset is now ready for categorical feature encoding

# Feature Encoding

Machine learning models require numerical input. The `type` column contains transaction categories (such as PAYMENT, TRANSFER, CASH_OUT, etc.), so it must be converted into numeric values before training the model.

In [18]:
# Display unique transaction types

print("Transaction Types:")
print(df["type"].unique())

Transaction Types:
<ArrowStringArray>
['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN']
Length: 5, dtype: str


In [19]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Encode transaction type
df["type"] = label_encoder.fit_transform(df["type"])

print("Encoding Completed Successfully!")

Encoding Completed Successfully!


In [20]:
# Display first 10 rows

df.head(10)

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,3,9839.64,170136.00,160296.36,0.0,0.00,0,0
1,1,3,1864.28,21249.00,19384.72,0.0,0.00,0,0
2,1,4,181.00,181.00,0.00,0.0,0.00,1,0
3,1,1,181.00,181.00,0.00,21182.0,0.00,1,0
4,1,3,11668.14,41554.00,29885.86,0.0,0.00,0,0
5,1,3,7817.71,53860.00,46042.29,0.0,0.00,0,0
6,1,3,7107.77,183195.00,176087.23,0.0,0.00,0,0
7,1,3,7861.64,176087.23,168225.59,0.0,0.00,0,0
8,1,3,4024.36,2671.00,0.00,0.0,0.00,0,0
9,1,2,5337.77,41720.00,36382.23,41898.0,40348.79,0,0


In [21]:
print(df.dtypes)

step                int64
type                int64
amount            float64
oldbalanceOrg     float64
newbalanceOrig    float64
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


## Summary

- Identified unique transaction categories.
- Encoded the `type` column using LabelEncoder.
- Converted all features into numerical format.
- Dataset is now ready for feature selection and train-test splitting.

# Feature Selection

The dataset is divided into:

- **Features (X):** Input variables used for prediction.
- **Target (y):** Output variable (`isFraud`) indicating whether a transaction is fraudulent.

The data is then split into training and testing sets to evaluate the model on unseen data.

In [23]:
# Create features (X) and target (y)

X = df.drop("isFraud", axis=1)
y = df["isFraud"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (6362077, 8)
Target Shape: (6362077,)


In [24]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Training Data Shape: (5089661, 8)
Testing Data Shape: (1272416, 8)


# Model Training

In this section, we train a Random Forest Classifier to identify fraudulent transactions.

Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve prediction accuracy and reduce overfitting.

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [26]:
# Create Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created successfully!")

Random Forest model created successfully!


In [27]:
# Train the model

rf_model.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [28]:
# Predict on test data

y_pred = rf_model.predict(X_test)

print("Predictions generated successfully!")

Predictions generated successfully!


In [29]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9997029273445163


In [30]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270777
           1       0.98      0.79      0.87      1639

    accuracy                           1.00   1272416
   macro avg       0.99      0.89      0.94   1272416
weighted avg       1.00      1.00      1.00   1272416



In [31]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1270750      27]
 [    351    1288]]


In [34]:
import joblib
import os

# Create models folder
os.makedirs("../models", exist_ok=True)

# Save trained Random Forest model
joblib.dump(rf_model, "../models/fraud_detection_model.pkl")

print("Fraud detection model saved successfully!")

Fraud detection model saved successfully!


In [33]:
%whos

Variable                 Type                      Data/Info
------------------------------------------------------------
DATA_PATH                str                       ../data/PS_20174392719_1491204439457_log.csv
LabelEncoder             type                      <class 'sklearn.preproces<...>ing._label.LabelEncoder'>
RandomForestClassifier   ABCMeta                   <class 'sklearn.ensemble.<...>.RandomForestClassifier'>
X                        DataFrame                 Shape: (6362077, 8)
X_test                   DataFrame                 Shape: (1272416, 8)
X_train                  DataFrame                 Shape: (5089661, 8)
accuracy_score           function                  <function accuracy_score at 0x000002AB4284D640>
classification_report    function                  <function classification_<...>rt at 0x000002AB4284EB90>
cm                       ndarray                   2x2: 4 elems, type `int64`, 32 bytes
confusion_matrix         function                  <function 